#  Interprétabilité & Analyse des Biais — Sections 6 / 7 / 8

**Projet 3 — Détection automatique de Fake News politiques**
**Groupe 3**

---

**Sections couvertes :**
- **6.1** Interprétabilité locale (LIME) et globale (top features TF-IDF)
- **6.2** Biais, fairness et risques
- **6.3** Limites des modèles
- **7** Discussion et perspectives
- **8** Conclusion + Références

In [3]:
import os, re, json, glob, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from lime.lime_text import LimeTextExplainer
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
DATA_DIR     = Path('../data')
TRAITEES_DIR = DATA_DIR / 'traitees'
DOC_DIR      = Path('../Doc')
DOC_DIR.mkdir(exist_ok=True)
for m_path in [DATA_DIR / 'modeles', Path('../models')]:
    if m_path.exists():
        MODELS_DIR = m_path
        break
STOP = set(stopwords.words('english'))
LABEL_ORDER  = ['pants-fire','false','barely-true','half-true','mostly-true','true']
LABEL_COLORS = ['#E24B4A','#F09595','#FAC775','#97C459','#5DCAA5','#1D9E75']
print(' Imports OK')

ModuleNotFoundError: No module named 'plotly'

In [ ]:
# Chargement données
try:
    test_df  = pd.read_parquet(TRAITEES_DIR / 'liar_test.parquet')
    train_df = pd.read_parquet(TRAITEES_DIR / 'liar_train.parquet')
except Exception as e:
    print(f'  {e}')
    test_df = train_df = None

bin_col = 'label_binary' if test_df is not None and 'label_binary' in test_df.columns else 'binary_label'
enr_col = 'enriched_text' if test_df is not None and 'enriched_text' in test_df.columns else 'statement'
print(f'Colonnes : bin={bin_col}, text={enr_col}')

# Chargement modèles
tfidf_files = glob.glob(str(MODELS_DIR / '*tfidf*'))
lr_files    = glob.glob(str(MODELS_DIR / '*logreg*'))
tfidf    = joblib.load(tfidf_files[0]) if tfidf_files else None
lr_model = joblib.load(lr_files[0])   if lr_files    else None

if tfidf and lr_model and test_df is not None:
    X_test = tfidf.transform(test_df[enr_col].fillna('').astype(str))
    y_test = test_df[bin_col]
    test_df['pred_lr'] = lr_model.predict(X_test)
    print(f' Modèles chargés | Test : {len(test_df)} déclarations')

## 6.1 Interprétabilité globale — Top features TF-IDF

> Les coefficients de la Logistic Regression révèlent quels mots le modèle associe à 'fake' ou 'real'.

In [ ]:
if tfidf and lr_model:
    try:
        coef = lr_model.coef_[0]
        feature_names = np.array(tfidf.get_feature_names_out())
    except AttributeError:
        try:
            coef = lr_model.named_steps['clf'].coef_[0]
            feature_names = np.array(lr_model.named_steps['tfidf'].get_feature_names_out())
        except Exception as e:
            print(f' {e}')
            coef = feature_names = None

    if coef is not None:
        top_n = 25
        top_fake = np.argsort(coef)[:top_n][::-1]
        top_real = np.argsort(coef)[-top_n:][::-1]

        # Plotly interactif
        df_fake = pd.DataFrame({'Feature': feature_names[top_fake], 'Score': np.abs(coef[top_fake])})
        df_real = pd.DataFrame({'Feature': feature_names[top_real], 'Score': np.abs(coef[top_real])})

        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=[f'Top {top_n} → FAKE', f'Top {top_n} → REAL'])
        fig.add_trace(go.Bar(y=df_fake['Feature'], x=df_fake['Score'],
                             orientation='h', marker_color='#E24B4A', name='FAKE'), row=1, col=1)
        fig.add_trace(go.Bar(y=df_real['Feature'], x=df_real['Score'],
                             orientation='h', marker_color='#1D9E75', name='REAL'), row=1, col=2)
        fig.update_layout(title='6.1 — Features discriminants (Logistic Regression)',
                          height=700, showlegend=False)
        fig.show()

        # Matplotlib pour rapport
        fig_mpl, axes = plt.subplots(1, 2, figsize=(16, 7))
        axes[0].barh(feature_names[top_fake][::-1], np.abs(coef[top_fake][::-1]),
                     color='#E24B4A', edgecolor='white')
        axes[0].set_title(f'Top {top_n} mots → FAKE', fontweight='bold', color='#A32D2D')
        axes[0].set_xlabel('|Coefficient|')
        axes[1].barh(feature_names[top_real][::-1], np.abs(coef[top_real][::-1]),
                     color='#1D9E75', edgecolor='white')
        axes[1].set_title(f'Top {top_n} mots → REAL', fontweight='bold', color='#0F6E56')
        axes[1].set_xlabel('|Coefficient|')
        plt.suptitle('Features discriminants — Logistic Regression (TF-IDF)', fontweight='bold')
        plt.tight_layout()
        plt.savefig(DOC_DIR / 'BIAS_01_top_features.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(' Doc/BIAS_01_top_features.png')

### 6.1.2 Interprétabilité locale — LIME

> LIME génère des explications locales : pourquoi le modèle a-t-il prédit 'fake' ou 'real' pour cette déclaration ?

In [ ]:
if tfidf and lr_model and test_df is not None:
    def predict_proba_fn(texts):
        try:
            return lr_model.predict_proba(tfidf.transform(texts))
        except:
            try:
                return lr_model.predict_proba(texts)
            except:
                vecs = tfidf.transform(texts)
                return lr_model.predict_proba(vecs)

    explainer = LimeTextExplainer(class_names=['fake', 'real'])

    cases = {
        'Correct FAKE ':  test_df[(test_df[bin_col]==0) & (test_df['pred_lr']==0)].index,
        'Correct REAL ':  test_df[(test_df[bin_col]==1) & (test_df['pred_lr']==1)].index,
        'Erreur FAKE ':   test_df[(test_df[bin_col]==0) & (test_df['pred_lr']==1)].index,
        'Erreur REAL ':   test_df[(test_df[bin_col]==1) & (test_df['pred_lr']==0)].index,
    }

    for case_name, idx_list in cases.items():
        if len(idx_list) == 0:
            continue
        row      = test_df.loc[idx_list[0]]
        text     = str(row[enr_col])
        true_lbl = 'fake' if row[bin_col]==0 else 'real'
        pred_lbl = 'fake' if row['pred_lr']==0 else 'real'

        print(f'\n─── {case_name} ────────────────────────')
        stmt_col = 'statement' if 'statement' in row.index else enr_col
        print(f'  Déclaration : {str(row.get(stmt_col, ""))[:90]}...')
        print(f'  Speaker : {row.get("speaker","N/A")} | Parti : {row.get("party","N/A")}')
        print(f'  Vrai : {true_lbl} | Prédit : {pred_lbl}')

        exp = explainer.explain_instance(text, predict_proba_fn, num_features=10, num_samples=400)
        print('  Mots influents :')
        for word, weight in exp.as_list():
            sym = '🔴 →fake' if weight < 0 else '🟢 →real'
            print(f'    {sym}  "{word}"  ({weight:+.4f})')

        html_path = DOC_DIR / f'LIME_{case_name[:12].replace(" ","_")}.html'
        exp.save_to_file(str(html_path))
        print(f'   {html_path}')

## 6.2 Biais, Fairness et Risques

In [ ]:
if test_df is not None and 'pred_lr' in test_df.columns:
    party_col = next((c for c in ['party','political_party'] if c in test_df.columns), None)
    if party_col:
        top_parties = test_df[test_df[party_col] != 'unknown'][party_col].value_counts().head(8).index
        party_res = []
        for party in top_parties:
            sub = test_df[test_df[party_col] == party]
            if len(sub) < 5: continue
            party_res.append({
                'Parti': party, 'N': len(sub),
                'Taux fake (%)': round((sub[bin_col]==0).mean()*100, 1),
                'F1 macro':  round(f1_score(sub[bin_col], sub['pred_lr'], average='macro', zero_division=0), 3),
                'Accuracy':  round(accuracy_score(sub[bin_col], sub['pred_lr']), 3),
            })
        bias_df = pd.DataFrame(party_res).sort_values('F1 macro', ascending=False)
        print(' Performances par parti politique :')
        display(bias_df)

        # Plotly
        fig = px.bar(bias_df, x='Parti', y='F1 macro',
                     color='Taux fake (%)', color_continuous_scale='RdYlGn_r',
                     title='6.2 — F1 macro et taux fake par parti politique',
                     text='F1 macro')
        fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
        fig.update_layout(yaxis_range=[0, 1])
        fig.show()

        # Matplotlib
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].bar(bias_df['Parti'], bias_df['F1 macro'], color='#7F77DD', edgecolor='white')
        axes[0].set_ylim(0, 1)
        axes[0].set_title('F1 macro par parti', fontweight='bold')
        axes[0].tick_params(axis='x', rotation=35)
        cols = ['#E24B4A' if r>60 else '#EF9F27' if r>40 else '#1D9E75'
                for r in bias_df['Taux fake (%)']]
        axes[1].bar(bias_df['Parti'], bias_df['Taux fake (%)'], color=cols, edgecolor='white')
        axes[1].axhline(50, color='gray', linestyle='--')
        axes[1].set_title('Taux fake par parti (%)', fontweight='bold')
        axes[1].set_ylim(0, 100)
        axes[1].tick_params(axis='x', rotation=35)
        plt.suptitle('Biais partisans', fontweight='bold')
        plt.tight_layout()
        plt.savefig(DOC_DIR / 'BIAS_02_party_bias.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(' Doc/BIAS_02_party_bias.png')

In [ ]:
if test_df is not None and 'speaker' in test_df.columns:
    spk = (
        test_df[test_df['speaker'] != 'unknown']
        .groupby('speaker')[bin_col]
        .agg(total='count', fake_count=lambda x: (x==0).sum())
        .query('total >= 5').copy()
    )
    spk['fake_rate'] = spk['fake_count'] / spk['total']
    spk = spk.sort_values('fake_rate', ascending=False).head(12)

    plt.figure(figsize=(13, 5))
    cols_spk = ['#E24B4A' if r>0.65 else '#EF9F27' if r>0.45 else '#1D9E75' for r in spk['fake_rate']]
    bars = plt.bar(spk.index, spk['fake_rate'], color=cols_spk, edgecolor='white')
    plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7)
    for bar, val, n in zip(bars, spk['fake_rate'], spk['total']):
        plt.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.0%}\n(n={n})',
                 ha='center', fontsize=8)
    plt.title('Taux fake par speaker (n≥5) — Biais de réputation', fontweight='bold')
    plt.ylabel('Taux fake')
    plt.xticks(rotation=40, ha='right')
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(DOC_DIR / 'BIAS_03_speaker_bias.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(' Doc/BIAS_03_speaker_bias.png')

In [ ]:
print('='*65)
print('  6.2 — RÉPONSES AUX 3 QUESTIONS ÉTHIQUES')
print('='*65)
print('''
① Le modèle favorise-t-il certains groupes ?
   Oui. Le modèle apprend des associations parti→fake
   qui peuvent refléter des biais du dataset PolitiFact
   plutôt que la réalité objective. Enrichir avec le
   speaker amplifie ce biais de réputation.

② Les erreurs ont-elles des conséquences réelles ?
   Oui, asymétriques :
   • FP (real→fake) : censure injustifiée
   • FN (fake→real) : désinformation non détectée
   Un système réel doit intégrer un seuil de confiance
   et une validation humaine pour les cas ambigus.

③ Peut-on détecter les fake news par le texte seul ?
   Non. Le NLP détecte des patterns linguistiques,
   pas des faits objectifs. Exemple : "Le chômage est
   à 4.2%" → impossible à vérifier sans données.
   Le texte seul est insuffisant pour :
   • Vérifier des faits chiffrés
   • Détecter l\'ironie/sarcasme (TF-IDF)
   • Comprendre le contexte historique
''')
print('='*65)

## 6.3 Limites des modèles et des approches

In [ ]:
if test_df is not None and 'pred_lr' in test_df.columns:
    errors  = test_df[test_df[bin_col] != test_df['pred_lr']].copy()
    correct = test_df[test_df[bin_col] == test_df['pred_lr']].copy()
    stmt_col = 'statement' if 'statement' in test_df.columns else enr_col
    errors['tlen']  = errors[stmt_col].fillna('').apply(lambda x: len(str(x).split()))
    correct['tlen'] = correct[stmt_col].fillna('').apply(lambda x: len(str(x).split()))

    print(f'Corrects : {len(correct)} ({len(correct)/len(test_df)*100:.1f}%)')
    print(f'Erreurs  : {len(errors)} ({len(errors)/len(test_df)*100:.1f}%)')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if 'label' in test_df.columns:
        err_lbl = errors['label'].value_counts().reindex(LABEL_ORDER).fillna(0)
        axes[0].bar(err_lbl.index, err_lbl.values, color=LABEL_COLORS, edgecolor='white')
        axes[0].set_title('Erreurs par label original', fontweight='bold')
        axes[0].set_xlabel('Label')
        axes[0].set_ylabel('Nb erreurs')
        axes[0].tick_params(axis='x', rotation=25)
    axes[1].hist(correct['tlen'], bins=30, alpha=0.7, color='#7F77DD',
                 label=f'Corrects (n={len(correct)})', edgecolor='white')
    axes[1].hist(errors['tlen'],  bins=30, alpha=0.7, color='#E24B4A',
                 label=f'Erreurs (n={len(errors)})', edgecolor='white')
    axes[1].set_title('Longueur : corrects vs erreurs', fontweight='bold')
    axes[1].set_xlabel('Nb mots')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(DOC_DIR / 'BIAS_04_error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(' Doc/BIAS_04_error_analysis.png')

## 7. Discussion et perspectives

## 8. Conclusion

In [ ]:
print('='*65)
print('  8. CONCLUSION')
print('='*65)
print('''
Ce projet a permis de construire un pipeline NLP complet
pour la détection automatique de fake news politiques.

APPORTS TECHNIQUES
→ Comparaison de 5 approches : TF-IDF, GloVe,
  DistilBERT, RoBERTa sur le LIAR Dataset
→ Gestion du déséquilibre (SMOTE, class_weight,
  WeightedRandomSampler, Focal Loss)
→ Évaluation out-of-domain sur BuzzFeed
→ Interprétabilité via LIME et coefficients LR

APPORTS CRITIQUES
→ Identification des biais partisans et de speaker
→ Démonstration que le texte seul est insuffisant
→ Analyse des conséquences asymétriques des erreurs

RÉSULTAT PRINCIPAL
→ RoBERTa (F1 ≈ 0.648) est le meilleur modèle
→ LR/TF-IDF (F1 ≈ 0.623) : meilleur rapport perf/coût
→ Le domain shift reste un défi majeur

La détection automatique de fake news est un outil
d\'aide à la décision, pas un système de censure.
La supervision humaine reste indispensable.
''')
print('='*65)

## Références

1. **Wang, W. Y. (2017)**. "Liar, Liar Pants on Fire": A new benchmark dataset for fake news detection. *ACL 2017*.
2. **Devlin et al. (2019)**. BERT: Pre-training of Deep Bidirectional Transformers. *NAACL-HLT*.
3. **Liu et al. (2019)**. RoBERTa: A Robustly Optimized BERT Pretraining Approach. *arXiv:1907.11692*.
4. **Pennington et al. (2014)**. GloVe: Global Vectors for Word Representation. *EMNLP*.
5. **BuzzFeed News (2016)**. Facebook Fact-Check Dataset. *GitHub*.
6. **Ribeiro et al. (2016)**. "Why Should I Trust You?": LIME. *KDD*.
7. LIAR Dataset — Kaggle / dataset-dev.
8. Articles récents — *MDPI, ACL Anthology*.

In [ ]:
print(' Projet terminé !')
print('\n Graphiques générés dans Doc/')
import os
for f in sorted(os.listdir(DOC_DIR)):
    size = os.path.getsize(DOC_DIR / f)
    print(f'  {f:<50} {size//1024:>4} Ko')